In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import DBSCAN
from pathlib import Path
import os
import hdbscan

In [ ]:
basenames = [
    "IHOPE14_MedLN_BottomLeft",
    "IHOPE14_MedLN_BottomRight",
    "IHOPE14_MedLN_TopRight",
    "IHOPE14_mesLN",
    "IHOPE20_LN",
    "IHOPE20_Spleen",
    "IHOPE26_LN",
    "IHOPE26_Spleen",
    "IHOPE27_LN",
    "IHOPE27_Spleen",
    "IHOPE39_LN",
    "IHOPE39_MesLN_1",
    "IHOPE39_Spleen"
]

DATA_DIR = Path("../data/processed/anndata/NEW")
OUT_DIR = Path("../results/reports/follicle_counts")
OUT_DIR.mkdir(exist_ok=True, parents=True)

In [ ]:
adata = sc.read_h5ad(
    DATA_DIR / "IHOPE14_MedLN_BottomLeft_celltypes_follicledomains.h5ad"
)

print(adata)
print(adata.obs.columns)
print(adata.obsm.keys())

In [ ]:
print((adata.obs["type_B"] == True).sum())
print((adata.obs["B_follicle"] == True).sum())

mask = (
    (adata.obs["type_B"] == True) &
    (adata.obs["B_follicle"] == True)
)

print(f"B cells inside follicle domains: {mask.sum()}")

In [ ]:
import scripts.follicle_counting_helpers as fh
import importlib
importlib.reload(fh)

results = []

for basename in basenames:
    try:
        print(f"\nProcessing {basename}")

        filepath = (
            DATA_DIR /
            f"{basename}_celltypes_follicledomains.h5ad"
        )

        adata = sc.read_h5ad(filepath)

        mask = (
            (adata.obs["type_B"] == True) &
            (adata.obs["B_follicle"] == True)
        )

        print(f"{basename}: {mask.sum()} B cells in follicle regions")

        adata, n_follicles = fh.detect_follicles(adata)

        fh.plot_follicles(adata, basename, OUT_DIR)

        print(f"Plotted {basename}, saved as {OUT_DIR / f'{basename}_follicles.png'}")

        adata.write(
            OUT_DIR / f"{basename}_follicle_clustered.h5ad"
        )

        results.append({
            "sample": basename,
            "n_follicles": n_follicles,
            "n_candidate_Bcells": mask.sum()
        })

    except Exception as e:
        print(f"Failed on {basename}: {e}")

In [ ]:
results_df = pd.DataFrame(results)

results_df.to_csv(
    OUT_DIR / "follicle_counts_summary.csv",
    index=False
)

results_df

In [ ]:
from scripts import follicle_counting_helpers as fh
import importlib
importlib.reload(fh)

results_hdb = []

for basename in basenames:
    try:
        print(f"\nProcessing {basename} (HDBSCAN all cells)")

        filepath = DATA_DIR / f"{basename}_celltypes_follicledomains.h5ad"
        adata = sc.read_h5ad(filepath)

        adata, n_clusters = fh.detect_follicles_hdbscan_all_cells(
            adata,
            min_cluster_size=200
        )

        results_hdb.append({
            "sample": basename,
            "n_clusters_hdb": n_clusters
        })

        # quick QC plot
        coords = adata.obsm["spatial"]

        plt.figure(figsize=(10, 10))

        sns.scatterplot(
            x=coords[:, 0],
            y=coords[:, 1],
            hue=adata.obs["follicle_cluster_hdb"].astype(str),
            palette="tab20",
            s=4,
            linewidth=0,
            legend=False
        )

        plt.title(f"{basename} - HDBSCAN all cells")
        plt.gca().invert_yaxis()
        plt.xticks([])
        plt.yticks([])
        plt.xlabel("")
        plt.ylabel("")
        plt.gca().set_aspect("equal", adjustable="box")

        plt.tight_layout()

        plt.savefig(
            OUT_DIR / f"{basename}_HDBSCAN_allcells.png",
            dpi=300
        )

        plt.show()
        plt.close()

    except Exception as e:
        print(f"Failed on {basename}: {e}")